In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load the feather file
df = pd.read_feather("../Data/EPISL_01_W1.feather")

print(f"Loaded {len(df)} SW events from feather file")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nShape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())
print(f"\nData info:")
print(df.info()) 

Loaded 3022727 SW events from feather file

Columns: ['wave_start', 'wave_stop', 'amp_peak', 'amp_trough', 'ndx_peak', 'ndx_trough', 'mid_xing', 'channel']

Shape: (3022727, 8)

Available channels: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(35), np.int64(36), np.int64(37), np.int64(38), np.int64(39), np.int64(40), np.int64(41), np.int64(43), np.int64(44), np.int64(45), np.int64(46), np.int64(49), np.int64(50), np.int64(51), np.int64(52), np.int64(53), np.int64(54), np.int64(56), np.int64(57), np.int64(58), np.int64(59), np.int64(60), np.int64(61),

In [ ]:
# Analyze the data structure
print(f"Number of unique channels: {df['channel'].nunique()}")
print(f"\nChannel distribution:")
print(df['channel'].value_counts().sort_index().head(20))
print(f"\nWave duration statistics (in samples):")
df['duration'] = df['wave_stop'] - df['wave_start']
print(df['duration'].describe())


Number of unique channels: 109

Channel distribution:
channel
0     26079
1     26598
2     27494
3     28659
4     29253
5     29452
6     29565
7     26356
8     27258
9     28099
10    28747
11    29047
12    29533
13    26706
14    27198
15    28380
16    27203
17    28399
18    28708
19    29211
Name: count, dtype: int64

Wave duration statistics (in samples):
count    3.022727e+06
mean     1.239057e+02
std      4.194393e+01
min      6.300000e+01
25%      8.800000e+01
50%      1.180000e+02
75%      1.530000e+02
max      2.500000e+02
Name: duration, dtype: float64

Channel 35 statistics:
  Number of events: 29021
  Duration range: 63-250 samples
  Index range: 59-3605419


In [ ]:
# Transform to GUI format: create SW events CSV with start_idx and stop_idx
# The GUI expects columns: start_idx, stop_idx
# We'll use wave_start -> start_idx and wave_stop -> stop_idx

sw_events = df[['wave_start', 'wave_stop']].copy()
sw_events.columns = ['start_idx', 'stop_idx']

# Remove duplicates if any (events that appear in multiple channels)
# Keep the first occurrence
sw_events = sw_events.drop_duplicates(subset=['start_idx', 'stop_idx'])

# Sort by start_idx for easier navigation
sw_events = sw_events.sort_values('start_idx').reset_index(drop=True)

print(f"Transformed to {len(sw_events)} unique SW events")
print(f"\nFirst few events:")
print(sw_events.head(10))
print(f"\nEvent index range: {sw_events['start_idx'].min()} to {sw_events['stop_idx'].max()}")


Filtering SW events for channel 35 to create global events...
Found 29021 SW events in channel 35

Global SW events (from channel 35): 29021

First few events:
   start_idx  stop_idx
0         59       149
1        149       385
2        889      1126
3       1427      1534
4       1534      1611
5       1611      1682
6       1682      1768
7       1768      1901
8       1901      2072
9       2072      2281

Event index range: 59 to 3605419


In [16]:
# Optional: Filter events by duration if needed
# Typical SW events are between 0.25-2 seconds
# Assuming 125 Hz sampling rate: 0.25s = 31 samples, 2s = 250 samples

sampling_rate = 125.0  # Adjust if your data has different sampling rate
min_duration_samples = int(0.25 * sampling_rate)  # 0.25 seconds
max_duration_samples = int(2.0 * sampling_rate)    # 2.0 seconds

sw_events['duration'] = sw_events['stop_idx'] - sw_events['start_idx']
filtered_sw_events = sw_events[
    (sw_events['duration'] >= min_duration_samples) & 
    (sw_events['duration'] <= max_duration_samples)
].copy()

print(f"Original events: {len(sw_events)}")
print(f"Filtered events (duration {min_duration_samples}-{max_duration_samples} samples): {len(filtered_sw_events)}")
print(f"\nDuration distribution of filtered events:")
print(filtered_sw_events['duration'].describe())


Original events: 29021
Filtered events (duration 31-250 samples): 29021

Duration distribution of filtered events:
count    29021.000000
mean       118.309810
std         39.345673
min         63.000000
25%         85.000000
50%        112.000000
75%        145.000000
max        250.000000
Name: duration, dtype: float64


In [17]:
# Save to CSV for use in GUI
# Remove the duration column before saving (GUI doesn't need it)
output_file = "../Data/EPISL_01_W1_sw_events.csv"

# Use filtered events or all events
events_to_save = filtered_sw_events[['start_idx', 'stop_idx']].copy()

events_to_save.to_csv(output_file, index=False)
print(f"Saved {len(events_to_save)} SW events to {output_file}")
print(f"\nCSV file preview:")
print(events_to_save.head(10))


Saved 29021 SW events to ../Data/EPISL_01_W1_sw_events.csv

CSV file preview:
   start_idx  stop_idx
0         59       149
1        149       385
2        889      1126
3       1427      1534
4       1534      1611
5       1611      1682
6       1682      1768
7       1768      1901
8       1901      2072
9       2072      2281


In [ ]:
# Optional: Create a version with channel information preserved
# This can be useful for analysis but GUI only needs start_idx and stop_idx

sw_events_with_channel = df[['wave_start', 'wave_stop', 'channel', 'amp_peak', 'amp_trough']].copy()
sw_events_with_channel.columns = ['start_idx', 'stop_idx', 'channel', 'amp_peak', 'amp_trough']

# Save detailed version (optional)
detailed_output = "../Data/EPISL_01_W1_sw_events_detailed.csv"
sw_events_with_channel.to_csv(detailed_output, index=False)
print(f"Saved detailed version with channel info to {detailed_output}")


Saved detailed version with channel 35 info to ../Data/EPISL_01_W1_sw_events_detailed.csv
  Contains 29021 events from channel 35


## Summary

This notebook:
1. Loads SW events from the feather file
2. Transforms `wave_start`/`wave_stop` to `start_idx`/`stop_idx` format
3. Filters events by duration (0.25-2 seconds, assuming 125 Hz sampling rate)
4. Saves to CSV format compatible with the GUI

**Output files:**
- `Data/EPISL_01_W1_sw_events.csv` - Simple format for GUI (start_idx, stop_idx)
- `Data/EPISL_01_W1_sw_events_detailed.csv` - Detailed format with channel info (optional)

**To use in GUI:**
```bash
python main.py --mat-file Data/EPISL_01_W1/EPISL_01_W1/EPISL_01_W1_EEG_FiltDwn_05to30Hz.mat --sw-csv Data/EPISL_01_W1_sw_events.csv
```
